# Data Integrity Audits

The Step 1 alignment audit is documented in [`selex_descriptor_alignment_audit.md`](selex_descriptor_alignment_audit.md). This notebook contains **Step 2: Descriptor Redundancy Audit**. It reads the combined SELEX tables but does not modify them or any previous notebook analyses.

## 2. Descriptor Redundancy Audit

We assess each descriptor group separately for tetramers and hexamers, then assess the complete stored numerical feature spaces. The group-level FCE definition follows the established 21 unique upper-triangle elements; the 15 mirrored lower-triangle columns that are also stored in the combined input tables are audited separately as exact duplicates.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Resolve paths as in the Step 1 data alignment audit
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SELEX_DIR = ROOT / "data" / "selex"
sys.path.insert(0, str(ROOT))

# Load the existing combined feature tables
input_tetramer = pd.read_csv(SELEX_DIR / "tetramer" / "input_tetramer.dat", sep=r"\s+")
input_hexamer = pd.read_csv(SELEX_DIR / "hexamer" / "input_hexamer.dat", sep=r"\s+")

# Keep snapshots so this audit can verify that source tables were not changed
input_tetramer_before = input_tetramer.copy(deep=True)
input_hexamer_before = input_hexamer.copy(deep=True)

print("Tetramer combined table:", input_tetramer.shape)
print("Hexamer combined table:", input_hexamer.shape)
print("Descriptor columns are numeric:",
      input_tetramer.drop(columns="SEQUENCE").select_dtypes(include=np.number).shape[1] == input_tetramer.shape[1] - 1,
      input_hexamer.drop(columns="SEQUENCE").select_dtypes(include=np.number).shape[1] == input_hexamer.shape[1] - 1)


### 2.1. Methods and numerical tolerance

Pearson and Spearman correlation matrices are calculated for each descriptor group. The rank analysis centers and scales columns to unit variance before applying SVD, so an exact affine dependency (the relevant notion when a model includes an intercept) is detected without descriptor scale affecting the rank. The numerical rank tolerance is `max(matrix dimensions) × machine epsilon × largest singular value`. A dependency is accepted only when its maximum reconstruction residual is at most `1e-10` relative to the feature magnitude; high correlations alone never trigger feature removal.

We report the full stored feature space as well as the canonical descriptor space using 21 FCE positions. Each exact dependency is expressed as an equation and logged with its group and maximum residual. Greedy retention follows the existing table/group order, so the earlier feature(s) are retained when an exact dependency is found.

In [ ]:
# Existing descriptor group definitions
DEGREES_OF_FREEDOM = ["SHIFT", "SLIDE", "RISE", "TILT", "ROLL", "TWIST"]
AVG_FEATURES = DEGREES_OF_FREEDOM.copy()
FCE_FEATURES = [
    f"{first}.{second}"
    for i, first in enumerate(DEGREES_OF_FREEDOM)
    for second in DEGREES_OF_FREEDOM[i:]
]
FCE_FEATURES_STORED = [
    f"{first}.{second}"
    for first in DEGREES_OF_FREEDOM
    for second in DEGREES_OF_FREEDOM
]
ELECTROSTATIC_FEATURES = ["e1", "e2", "e3", "e4", "e5"]
ION_FEATURES = {
    "tetramer": ["m1", "m2", "m3", "j1", "j2", "j3"],
    "hexamer": ["m0", "m1", "m2", "m3", "m4", "j0", "j1", "j2", "j3", "j4"],
}
DATASETS = {"tetramer": input_tetramer, "hexamer": input_hexamer}
GROUPS = {
    size: {
        "AVG": AVG_FEATURES,
        "FCE": FCE_FEATURES,
        "Electrostatic": ELECTROSTATIC_FEATURES,
        "Ion-related": ION_FEATURES[size],
    }
    for size in DATASETS
}
GROUP_ORDER = ["AVG", "FCE", "Electrostatic", "Ion-related"]
RANK_RESIDUAL_TOLERANCE = 1e-10


def standardized_centered_matrix(data, features):
    """Center and scale columns for a scale-invariant SVD rank calculation."""
    values = data.loc[:, features].to_numpy(dtype=float)
    means = values.mean(axis=0)
    scales = values.std(axis=0, ddof=0)
    scales[scales == 0] = 1.0
    return (values - means) / scales


def matrix_rank_details(data, features):
    """Return SVD rank and its standard floating-point numerical tolerance."""
    matrix = standardized_centered_matrix(data, features)
    singular_values = np.linalg.svd(matrix, compute_uv=False)
    largest = singular_values[0] if len(singular_values) else 0.0
    tolerance = max(matrix.shape) * np.finfo(float).eps * largest
    rank = int(np.sum(singular_values > tolerance))
    return {
        "feature_count": len(features),
        "rank": rank,
        "independent_dimensions": rank,
        "svd_tolerance": tolerance,
        "smallest_singular_values": singular_values[-3:].tolist(),
    }


def fit_exact_dependency(data, target, predictors):
    """Fit target from prior features plus intercept and quantify the residual."""
    target_values = data[target].to_numpy(dtype=float)
    if not predictors:
        intercept = float(target_values.mean())
        coefficients = np.array([], dtype=float)
        prediction = np.full_like(target_values, intercept)
    else:
        predictor_values = data[predictors].to_numpy(dtype=float)
        means = predictor_values.mean(axis=0)
        scales = predictor_values.std(axis=0, ddof=0)
        scales[scales == 0] = 1.0
        standardized = (predictor_values - means) / scales
        design = np.column_stack([np.ones(len(data)), standardized])
        fitted = np.linalg.lstsq(design, target_values, rcond=None)[0]
        coefficients = fitted[1:] / scales
        intercept = float(fitted[0] - means @ coefficients)
        prediction = intercept + predictor_values @ coefficients

    residual = target_values - prediction
    max_abs_residual = float(np.max(np.abs(residual)))
    relative_residual = max_abs_residual / max(1.0, float(np.max(np.abs(target_values))))
    return intercept, coefficients, max_abs_residual, relative_residual


def format_dependency_equation(target, intercept, coefficients, predictors):
    """Format a concise equation, omitting numerically zero terms."""
    pieces = []
    if abs(intercept) > 1e-8:
        pieces.append(f"{intercept:.8g}")
    for coefficient, feature in zip(coefficients, predictors):
        if abs(coefficient) <= 1e-8:
            continue
        magnitude = abs(coefficient)
        term = feature if np.isclose(magnitude, 1.0, atol=1e-8) else f"{magnitude:.8g} × {feature}"
        if not pieces:
            pieces.append(term if coefficient > 0 else f"−{term}")
        else:
            pieces.append((" + " if coefficient > 0 else " − ") + term)
    return f"{target} = " + ("".join(pieces) if pieces else "0")


def scan_exact_dependencies(data, features, group_by, scope_label):
    """Retain independent columns in order and record only residual-confirmed dependencies."""
    retained = []
    dependencies = []
    current_rank = 0
    for feature in features:
        candidate = retained + [feature]
        candidate_rank = matrix_rank_details(data, candidate)["rank"]
        if candidate_rank > current_rank:
            retained.append(feature)
            current_rank = candidate_rank
            continue

        intercept, coefficients, max_residual, relative_residual = fit_exact_dependency(
            data, feature, retained
        )
        if relative_residual > RANK_RESIDUAL_TOLERANCE:
            raise ValueError(
                f"{feature} appears rank-dependent but reconstruction residual "
                f"{relative_residual:.3g} exceeds tolerance."
            )
        used_predictors = [
            predictor for predictor, coefficient in zip(retained, coefficients)
            if abs(coefficient) > 1e-8
        ]
        used_groups = {group_by[predictor] for predictor in used_predictors}
        feature_group = group_by[feature]
        dependency_scope = (
            "between groups" if any(group != feature_group for group in used_groups)
            else "within group"
        )
        dependencies.append({
            "Feature involved": feature,
            "Descriptor group": feature_group,
            "Dependency equation": format_dependency_equation(
                feature, intercept, coefficients, retained
            ),
            "Within/between groups": dependency_scope,
            "Analysis scope": scope_label,
            "Rank before": current_rank,
            "Rank after": candidate_rank,
            "Maximum absolute residual": max_residual,
            "Relative residual": relative_residual,
            "Recommended feature(s) to retain": ", ".join(used_predictors),
            "Recommended feature to exclude": feature,
        })
    return retained, dependencies


def correlation_pair_summary(correlation, group_name, method):
    """List perfect and other strong pairwise correlations without removing features."""
    rows = []
    features = correlation.columns.tolist()
    for i, feature_1 in enumerate(features):
        for feature_2 in features[i + 1:]:
            value = float(correlation.loc[feature_1, feature_2])
            if abs(value) < 0.7:
                continue
            is_perfect = np.isclose(abs(value), 1.0, rtol=0, atol=1e-12)
            rows.append({
                "Group": group_name,
                "Method": method.title(),
                "Feature 1": feature_1,
                "Feature 2": feature_2,
                "Correlation": value,
                "Relationship": "Perfect pairwise correlation" if is_perfect else "Strong, imperfect correlation",
            })
    return rows


CORRELATION_MATRICES = {}
rank_rows = []
strong_correlation_rows = []
perfect_pair_rows = []
within_group_dependencies = []
full_dependencies = {}
full_rank_rows = []
clean_feature_lists = {}

for size, data in DATASETS.items():
    if "SEQUENCE" not in data.columns:
        raise KeyError(f"SEQUENCE missing from {size} input table.")
    numeric_columns = data.drop(columns="SEQUENCE").columns.tolist()
    if not all(pd.api.types.is_numeric_dtype(data[column]) for column in numeric_columns):
        raise TypeError(f"Non-numeric descriptor columns found in {size} table.")
    if len(numeric_columns) != len(set(numeric_columns)):
        raise ValueError(f"Repeated descriptor column names in {size} table.")

    group_by = {
        feature: group_name
        for group_name, features in GROUPS[size].items()
        for feature in features
    }
    # The combined tables store all 36 FCE positions; use the same 21-element
    # upper-triangle feature definition as the preceding EDA/PCA analyses.
    for feature in FCE_FEATURES_STORED:
        group_by[feature] = "FCE"

    for group_name in GROUP_ORDER:
        features = GROUPS[size][group_name]
        group_data = data.loc[:, features].copy()
        pearson = group_data.corr(method="pearson")
        spearman = group_data.corr(method="spearman")
        CORRELATION_MATRICES[(size, group_name, "pearson")] = pearson
        CORRELATION_MATRICES[(size, group_name, "spearman")] = spearman
        rank_details = matrix_rank_details(group_data, features)
        rank_rows.append({
            "Sequence length": size.title(),
            "Descriptor group": group_name,
            "Feature count": rank_details["feature_count"],
            "Matrix rank (centered/scaled)": rank_details["rank"],
            "Independent dimensions": rank_details["independent_dimensions"],
            "SVD tolerance": rank_details["svd_tolerance"],
        })
        for method, correlation in [("pearson", pearson), ("spearman", spearman)]:
            pairs = correlation_pair_summary(correlation, group_name, method)
            strong_correlation_rows.extend(
                {"Sequence length": size.title(), **pair} for pair in pairs
            )
            perfect_pair_rows.extend(
                {"Sequence length": size.title(), **pair}
                for pair in pairs if pair["Relationship"] == "Perfect pairwise correlation"
            )

        # Search exact dependencies within the canonical feature definition.
        _, dependencies = scan_exact_dependencies(
            group_data, features, group_by, f"within {group_name}"
        )
        within_group_dependencies.extend(
            {"Sequence length": size.title(), **dependency}
            for dependency in dependencies
        )

    # Audit all stored FCE positions, including their mirrored entries.
    _, stored_fce_dependencies = scan_exact_dependencies(
        data, FCE_FEATURES_STORED, group_by, "stored FCE matrix"
    )
    stored_fce_rank = matrix_rank_details(data, FCE_FEATURES_STORED)
    rank_rows.append({
        "Sequence length": size.title(),
        "Descriptor group": "FCE (all stored positions)",
        "Feature count": stored_fce_rank["feature_count"],
        "Matrix rank (centered/scaled)": stored_fce_rank["rank"],
        "Independent dimensions": stored_fce_rank["independent_dimensions"],
        "SVD tolerance": stored_fce_rank["svd_tolerance"],
    })
    within_group_dependencies.extend(
        {"Sequence length": size.title(), **dependency}
        for dependency in stored_fce_dependencies
    )

    # Canonical feature space: FCE upper triangle only, all other groups once.
    canonical_features = [
        feature for group_name in GROUP_ORDER for feature in GROUPS[size][group_name]
    ]
    canonical_group_by = {
        feature: group_name
        for group_name in GROUP_ORDER
        for feature in GROUPS[size][group_name]
    }
    canonical_rank = matrix_rank_details(data, canonical_features)
    group_rank_sum = sum(
        matrix_rank_details(data, GROUPS[size][group_name])["rank"]
        for group_name in GROUP_ORDER
    )
    additional_between_group_dependencies = group_rank_sum - canonical_rank["rank"]
    full_rank_rows.append({
        "Sequence length": size.title(),
        "Feature space": "Canonical groups (21 unique FCE)",
        "Feature count": len(canonical_features),
        "Matrix rank (centered/scaled)": canonical_rank["rank"],
        "Independent dimensions": canonical_rank["independent_dimensions"],
        "SVD tolerance": canonical_rank["svd_tolerance"],
        "Additional between-group dependencies": additional_between_group_dependencies,
    })

    full_rank = matrix_rank_details(data, numeric_columns)
    full_rank_rows.append({
        "Sequence length": size.title(),
        "Feature space": "All stored numeric columns",
        "Feature count": len(numeric_columns),
        "Matrix rank (centered/scaled)": full_rank["rank"],
        "Independent dimensions": full_rank["independent_dimensions"],
        "SVD tolerance": full_rank["svd_tolerance"],
        "Additional between-group dependencies": additional_between_group_dependencies,
    })
    clean_features, dependencies = scan_exact_dependencies(
        data, numeric_columns, group_by, "all stored numeric columns"
    )
    full_dependencies[size] = dependencies
    if len(clean_features) != full_rank["rank"]:
        raise AssertionError(f"Clean feature count and full matrix rank differ for {size}.")
    clean_feature_lists[size] = clean_features

rank_summary = pd.DataFrame(rank_rows)
full_space_summary = pd.DataFrame(full_rank_rows)
strong_correlations = pd.DataFrame(strong_correlation_rows)
perfect_correlations = pd.DataFrame(perfect_pair_rows)
within_group_dependency_table = pd.DataFrame(within_group_dependencies)
full_dependency_tables = {
    size: pd.DataFrame(dependencies) for size, dependencies in full_dependencies.items()
}

print("Within-group and FCE stored-position rank summary:")
display(rank_summary.round({"SVD tolerance": 12}))
print("Full feature-space ranks and between-group dependency check:")
display(full_space_summary.round({"SVD tolerance": 12}))
print("Perfect pairwise correlations (shown for Pearson and Spearman):")
display(perfect_correlations.round({"Correlation": 6}))
print("Other strong correlations (|r| ≥ 0.7; not treated as exact dependencies):")
display(
    strong_correlations[
        strong_correlations["Relationship"] == "Strong, imperfect correlation"
    ].sort_values(["Sequence length", "Group", "Method", "Correlation"], ignore_index=True)
    .round({"Correlation": 4})
)
print("Exact dependencies by sequence length, including mirrored FCE entries:")
for size, dependency_table in full_dependency_tables.items():
    print(f"\n{size.title()}:")
    display(dependency_table.round({"Maximum absolute residual": 12, "Relative residual": 12}))


### 2.2. Clean feature lists, removal log and validation

The recommended feature lists retain one representative from each exact dependency. Mirrored FCE entries use their upper-triangle counterpart; electrostatic features `e3` and `e5` are excluded using the equations shown in the removal log. Features with only strong, imperfect correlation remain in the lists.

The next cell imports the canonical definitions from `src/feature_config.py`, checks that the notebook results match them, verifies every retained feature exists and is numeric, and confirms that each clean matrix has full centered/scaled rank. The original DataFrames are also compared with the snapshots made at load time.

In [ ]:
from src.feature_config import (
    FEATURES_CLEAN_TETRAMER,
    FEATURES_CLEAN_HEXAMER,
    REMOVAL_LOG_TETRAMER,
    REMOVAL_LOG_HEXAMER,
    DEPENDENCY_RELATIVE_RESIDUAL_TOLERANCE,
)

expected_clean = {
    "tetramer": FEATURES_CLEAN_TETRAMER,
    "hexamer": FEATURES_CLEAN_HEXAMER,
}
expected_removal_logs = {
    "tetramer": REMOVAL_LOG_TETRAMER,
    "hexamer": REMOVAL_LOG_HEXAMER,
}
assert RANK_RESIDUAL_TOLERANCE == DEPENDENCY_RELATIVE_RESIDUAL_TOLERANCE
validation_rows = []
for size, data in DATASETS.items():
    clean_features = expected_clean[size]
    missing_features = sorted(set(clean_features) - set(data.columns))
    non_numeric = [
        feature for feature in clean_features
        if feature in data.columns and not pd.api.types.is_numeric_dtype(data[feature])
    ]
    clean_rank = matrix_rank_details(data, clean_features)
    expected_removals = {entry["feature"] for entry in expected_removal_logs[size]}
    actual_removals = {entry["Recommended feature to exclude"] for entry in full_dependencies[size]}
    assert not missing_features, (size, missing_features)
    assert not non_numeric, (size, non_numeric)
    assert clean_features == clean_feature_lists[size], size
    assert clean_rank["rank"] == len(clean_features), (size, clean_rank)
    assert actual_removals == expected_removals, (size, actual_removals, expected_removals)
    validation_rows.append({
        "Sequence length": size.title(),
        "Original stored numeric features": len(data.columns) - 1,
        "Exact dependencies removed": len(actual_removals),
        "Retained clean features": len(clean_features),
        "Clean matrix rank": clean_rank["rank"],
        "All names exist and are numeric": True,
        "No exact dependency remains": True,
    })

assert input_tetramer.equals(input_tetramer_before)
assert input_hexamer.equals(input_hexamer_before)
validation_rows.append({
    "Sequence length": "Both",
    "Original stored numeric features": "—",
    "Exact dependencies removed": "—",
    "Retained clean features": "—",
    "Clean matrix rank": "—",
    "All names exist and are numeric": "—",
    "No exact dependency remains": "—",
    "Original tables unchanged": True,
})
validation_summary = pd.DataFrame(validation_rows)
print("Canonical clean feature list validation:")
display(validation_summary)

# The removal log is available in both this notebook and the reusable module.
removal_log_tetramer = pd.DataFrame(REMOVAL_LOG_TETRAMER)
removal_log_hexamer = pd.DataFrame(REMOVAL_LOG_HEXAMER)
print("Tetramer removal log:")
display(removal_log_tetramer)
print("Hexamer removal log:")
display(removal_log_hexamer)


### 2.3. Audit findings

**Observed results from the stored combined tables:**

| Sequence length | Numeric columns stored | Canonical features before electrostatic exclusions | Full stored matrix rank | Exact dependencies removed | Clean features retained |
|---|---:|---:|---:|---:|---:|
| Tetramer | 53 | 38 | 36 | 17 | 36 |
| Hexamer | 57 | 42 | 40 | 17 | 40 |

The 17 exclusions per length consist of **15 mirrored FCE entries** and **two electrostatic identities**. The FCE group has rank 21 for its 21 unique upper-triangle descriptors; its 36 stored entries have rank 21 because each lower-triangle value duplicates its symmetric upper-triangle partner. Electrostatics has 5 features but rank 3 in both lengths. AVG has rank 6; ion-related descriptors have rank 6 for tetramers and 10 for hexamers. The canonical full feature ranks are 36 and 40, exactly equal to the sums of the within-group ranks, so **no additional exact dependencies between descriptor groups were detected**.

The electrostatic identities are `e3 = −e2` and `e5 = −e1 − e4`. The highest within-group imperfect Pearson correlations include `SLIDE.RISE`–`RISE.ROLL` in FCE (`r = 0.8956` tetramers; `r = 0.8870` hexamers), `SHIFT`–`TILT` in AVG (`r = 0.7989`; `r = 0.7928`), and `e1`–`e4` in electrostatics (`r = −0.8182` for both). In each case, high but imperfect correlation is reported, not treated as an exact dependency. Tetramer ion descriptors have three additional strong Spearman pairs (`m1`–`j1`, `m3`–`j3`, `m2`–`j2`); no ion pair reaches the threshold for hexamers.

The canonical lists and per-length removal logs are defined in [`src/feature_config.py`](../src/feature_config.py). They retain 36 tetramer and 40 hexamer features. Numerical dependency equations have residuals at or below approximately `1.1 × 10⁻¹⁴`, comfortably below the `1e-10` relative residual cutoff. The SVD rank threshold follows the dimension-scaled machine-precision rule stated above. This audit establishes dependencies in the values as stored; it cannot make claims about precision lost before those values were written to the input files.
